# Casting Defect Detection with a CNN 🏭
**Goal:** look at a photo of a metal casting and decide: **defective** (`def_front`) or **ok** (`ok_front`).

Steps:
1. Explore the data
2. Transforms → Dataset → DataLoader
3. Build a CNN model
4. Loss + optimizer
5. Train / test loop
6. Evaluate (curves, confusion matrix, predictions)
7. Save the model

## Step 1 – Explore the data

In [ ]:
from pathlib import Path 

data_dir=Path("data/casting_data")

for split in ["train" ,"test"]:
    for cls in ["def_front","ok_front"]:
        n=len(list((data_dir/ split /cls).glob("*.jpeg")))
        print(f"{split}  {cls}  {n} images")

In [ ]:
import os 

def walk_through(dir_path):
    for dirpath,dirnames,filenames in os.walk(dir_path):
      print(f" there are {len(dirnames)} directories and {len(filenames)} images in '{dirpath}' ")


walk_through(data_dir)

In [ ]:
import random 
import numpy as np
import matplotlib.pyplot as plt 
from PIL import Image 
import torch 
import torchvision 
from torchvision import datasets 

In [ ]:
# Get all image paths 

image_paths=list(data_dir.glob("*/*/*.jpeg"))

# Get random path 
random_path = random.choice(image_paths)

# Get the label 
label=random_path.parent.name 

# Open the Image 
img=Image.open(random_path)
print(f"Path:  {random_path}")
print(f"Label: {label}")
print(f"Size:  {img.size}  Mode: {img.mode}")

In [ ]:
img_array = np.array(img)
print(f"Shape: {img_array.shape}")
print(f"Dtype: {img_array.dtype}  Min: {img_array.min()}  Max: {img_array.max()}")

red   = img_array[:, :, 0]
green = img_array[:, :, 1]
blue  = img_array[:, :, 2]
print(np.array_equal(red, green), np.array_equal(green, blue))

## Step 2 – Transforms, Dataset and DataLoader

In [ ]:
train_path=data_dir/ "train"
test_path=data_dir/ "test"
train_path,test_path

In [ ]:
from torchvision import transforms , datasets 
data_transform = transforms.Compose([
transforms.Grayscale(),
transforms.Resize(size=(64,64)),
transforms.ToTensor()

])

In [ ]:
train_data=datasets.ImageFolder(root=train_path,transform=data_transform)
test_data =datasets.ImageFolder(root=test_path,transform=data_transform)
train_data,test_data

In [ ]:
from torch.utils.data import DataLoader
BATCH_SIZE=32
train_data_loader=DataLoader(dataset=train_data,batch_size=BATCH_SIZE ,shuffle=True ,num_workers=0)
test_data_loader=DataLoader(dataset=test_data,batch_size=BATCH_SIZE,shuffle=False,num_workers=0)

In [ ]:
img,label=next(iter(train_data_loader))
img,label

In [ ]:
label.shape

In [ ]:
train_data.classes ,train_data.class_to_idx

In [ ]:
image,target=train_data[0]
image ,target

## Step 3 – Build the CNN model
Two "pattern finder" blocks (Conv → ReLU → Conv → ReLU → MaxPool), then a classifier.

Shapes: `[B,1,64,64] → [B,10,32,32] → [B,10,16,16] → [B,2560] → [B,2]`

In [ ]:
from torch import nn

# Use the GPU if we have one, otherwise the CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


class CastingCNN(nn.Module):
    def __init__(self, in_channels=1, hidden_units=10, num_classes=2):
        super().__init__()

        # Block 1: find simple patterns (edges, spots)
        self.block_1 = nn.Sequential(
            nn.Conv2d(in_channels, hidden_units, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),  # halves height and width: 64 -> 32
        )

        # Block 2: combine simple patterns into bigger ones (cracks, holes)
        self.block_2 = nn.Sequential(
            nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),  # 32 -> 16
        )

        # Classifier: turn the feature maps into 2 scores (def / ok)
        self.classifier = nn.Sequential(
            nn.Flatten(),                                   # [B, 10, 16, 16] -> [B, 2560]
            nn.Linear(hidden_units * 16 * 16, num_classes)  # [B, 2560] -> [B, 2]
        )

    def forward(self, x):            # x shape: [B, 1, 64, 64]
        x = self.block_1(x)          # shape:   [B, 10, 32, 32]
        x = self.block_2(x)          # shape:   [B, 10, 16, 16]
        x = self.classifier(x)       # shape:   [B, 2]
        return x


torch.manual_seed(42)
model = CastingCNN(in_channels=1, hidden_units=10, num_classes=len(train_data.classes)).to(device)
model

In [ ]:
# Smoke test: send ONE real batch through the (untrained) model to check the shapes
img_batch, label_batch = next(iter(train_data_loader))
img_batch = img_batch.to(device)          # model and data must be on the same device!

model.eval()
with torch.inference_mode():
    logits = model(img_batch)

print(f"Input shape:  {img_batch.shape}")   # [32, 1, 64, 64]
print(f"Output shape: {logits.shape}")      # [32, 2]
print(f"Probabilities of first image: {torch.softmax(logits[0], dim=0)}")  # ~50/50, it knows nothing yet

## Step 4 – Loss function, optimizer and accuracy
- **CrossEntropyLoss**: the standard loss for classification. It punishes the model a lot when it is *confident and wrong*.
- **Adam** optimizer: updates the weights. `lr=0.001` is a safe default for Adam.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)


def accuracy_fn(y_true, y_pred):
    """Percentage of predictions that are correct."""
    correct = (y_true == y_pred).sum().item()
    return correct / len(y_true) * 100

## Step 5 – Train step and test step
**Train step** = the 5-step recipe for every batch:
1. forward pass → 2. loss → 3. zero grad → 4. backward → 5. optimizer step

**Test step** = only forward pass + loss, no learning (`inference_mode`).

In [ ]:
def train_step(model, dataloader, loss_fn, optimizer, device):
    model.train()                              # training mode
    train_loss, train_acc = 0, 0

    for X, y in dataloader:                    # X: [32, 1, 64, 64], y: [32]
        X, y = X.to(device), y.to(device)

        y_logits = model(X)                    # 1. forward pass   -> [32, 2]
        loss = loss_fn(y_logits, y)            # 2. calculate loss (a single number)
        optimizer.zero_grad()                  # 3. clear old gradients
        loss.backward()                        # 4. backpropagation
        optimizer.step()                       # 5. update the weights

        train_loss += loss.item()
        y_pred = y_logits.argmax(dim=1)        # [32, 2] -> [32] (index of the bigger score)
        train_acc += accuracy_fn(y, y_pred)

    # average over all batches
    return train_loss / len(dataloader), train_acc / len(dataloader)


def test_step(model, dataloader, loss_fn, device):
    model.eval()                               # evaluation mode
    test_loss, test_acc = 0, 0

    with torch.inference_mode():               # no gradients -> faster, less memory
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            y_logits = model(X)
            test_loss += loss_fn(y_logits, y).item()
            test_acc += accuracy_fn(y, y_logits.argmax(dim=1))

    return test_loss / len(dataloader), test_acc / len(dataloader)

## Step 6 – The training loop

In [ ]:
from tqdm.auto import tqdm   # nice progress bar


def train(model, train_dataloader, test_dataloader, loss_fn, optimizer, epochs, device):
    # a dictionary to remember the results of every epoch (for plotting later)
    results = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model, train_dataloader, loss_fn, optimizer, device)
        test_loss, test_acc = test_step(model, test_dataloader, loss_fn, device)

        print(f"Epoch {epoch+1:2d} | train loss: {train_loss:.4f} | train acc: {train_acc:.2f}% "
              f"| test loss: {test_loss:.4f} | test acc: {test_acc:.2f}%")

        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)

    return results

In [ ]:
from timeit import default_timer as timer

torch.manual_seed(42)
NUM_EPOCHS = 10

start = timer()
results = train(model, train_data_loader, test_data_loader, loss_fn, optimizer, NUM_EPOCHS, device)
print(f"Training time: {timer() - start:.1f} seconds")

## Step 7 – Loss and accuracy curves
- Both losses go **down** → the model is learning ✅
- Train loss goes down but test loss goes **up** → overfitting ⚠️

In [ ]:
epochs_range = range(1, len(results["train_loss"]) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, results["train_loss"], label="train")
plt.plot(epochs_range, results["test_loss"], label="test")
plt.title("Loss")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, results["train_acc"], label="train")
plt.plot(epochs_range, results["test_acc"], label="test")
plt.title("Accuracy (%)")
plt.xlabel("Epoch")
plt.legend()

plt.show()

## Step 8 – Confusion matrix
Accuracy alone can hide problems. The confusion matrix shows **which** mistakes the model makes.

In a factory, the worst mistake is calling a **defective** part "ok" (it gets shipped to a customer!).

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

# Collect predictions for the whole test set
all_preds, all_labels = [], []
model.eval()
with torch.inference_mode():
    for X, y in test_data_loader:
        y_logits = model(X.to(device))
        all_preds.append(y_logits.argmax(dim=1).cpu())  # back to CPU for sklearn
        all_labels.append(y)

all_preds = torch.cat(all_preds)    # shape: [715]
all_labels = torch.cat(all_labels)  # shape: [715]

cm = confusion_matrix(all_labels, all_preds)
ConfusionMatrixDisplay(cm, display_labels=train_data.classes).plot(cmap="Blues")
plt.title("Confusion matrix (test set)")
plt.show()

print(classification_report(all_labels, all_preds, target_names=train_data.classes, digits=3))

## Step 9 – Look at some predictions

In [ ]:
def predict_image(model, image_tensor, device):
    """image_tensor shape: [1, 64, 64] -> returns (predicted class index, probability)"""
    model.eval()
    with torch.inference_mode():
        x = image_tensor.unsqueeze(0).to(device)    # add batch dim: [1, 64, 64] -> [1, 1, 64, 64]
        probs = torch.softmax(model(x), dim=1)      # [1, 2]
    pred = probs.argmax(dim=1).item()
    return pred, probs[0, pred].item()


random.seed(42)
indices = random.sample(range(len(test_data)), k=9)

plt.figure(figsize=(9, 9))
for i, idx in enumerate(indices):
    image, true_label = test_data[idx]               # image shape: [1, 64, 64]
    pred, prob = predict_image(model, image, device)

    plt.subplot(3, 3, i + 1)
    plt.imshow(image.squeeze(), cmap="gray")         # [1, 64, 64] -> [64, 64] for matplotlib
    color = "green" if pred == true_label else "red"
    plt.title(f"Pred: {train_data.classes[pred]} ({prob:.2f})\nTrue: {train_data.classes[true_label]}",
              color=color, fontsize=10)
    plt.axis("off")
plt.tight_layout()
plt.show()

## Step 10 – Save and load the model
We save only the **state_dict** (the learned weights), not the whole object. This is the recommended way.

In [ ]:
model_dir = Path("models")
model_dir.mkdir(exist_ok=True)
model_path = model_dir / "casting_cnn.pth"

torch.save(model.state_dict(), model_path)
print(f"Saved model to: {model_path}")

# Load it back into a NEW model and check it gives the same results
loaded_model = CastingCNN(in_channels=1, hidden_units=10, num_classes=2).to(device)
loaded_model.load_state_dict(torch.load(model_path, weights_only=True))

loaded_loss, loaded_acc = test_step(loaded_model, test_data_loader, loss_fn, device)
print(f"Loaded model test acc: {loaded_acc:.2f}%  (should match the last epoch)")